# 01 — CairoGo Dataset Overview

**Pack:** v17 | **Leader ID:** 20231067  
**Goal:** Phase 0.2 — Dataset inspection. No cleaning or modelling here.

> ⚠️ Read `docs/DATA_DICTIONARY.md` and `docs/PROJECT_DECISIONS.md` before editing this notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)

PACK = 'data/raw/cairogo_pack'
FOUNDATION = 'data/raw/foundation'
RANDOM_STATE = 42
print('Libraries loaded ✓')

## 1. Load All Tables

In [ ]:
trips    = pd.read_csv(f'{PACK}/trips.csv', parse_dates=['request_ts'])
customers = pd.read_csv(f'{PACK}/customers.csv', parse_dates=['signup_date'])
drivers  = pd.read_csv(f'{PACK}/drivers.csv', parse_dates=['join_date'])
zones    = pd.read_csv(f'{PACK}/zones.csv')
recent   = pd.read_csv(f'{PACK}/recent_labelled.csv', parse_dates=['request_ts'])
holdout  = pd.read_csv(f'{PACK}/holdout_inputs.csv', parse_dates=['request_ts'])

tables = {'trips': trips, 'customers': customers, 'drivers': drivers,
          'zones': zones, 'recent_labelled': recent, 'holdout_inputs': holdout}

for name, df in tables.items():
    print(f'{name:20s}: {df.shape[0]:6,} rows × {df.shape[1]:2} cols')

## 2. trips.csv — Shape, Types & Sample

In [ ]:
print('Shape:', trips.shape)
trips.head(3)

In [ ]:
trips.dtypes

## 3. Missing Values

In [ ]:
missing = trips.isnull().sum()
missing_pct = (missing / len(trips) * 100).round(2)
miss_df = pd.DataFrame({'count': missing, 'pct': missing_pct})
miss_df = miss_df[miss_df['count'] > 0]
print('Columns with missing values:')
print(miss_df.to_string())
print('\n⚠️  actual_trip_time_min, cancel_reason, driver_arrival_min are missing ONLY for failed trips — this is expected (not a data quality issue).')

## 4. Target Variable — request_failed

In [ ]:
target_counts = trips['request_failed'].value_counts()
target_pct    = trips['request_failed'].value_counts(normalize=True) * 100

print('Target distribution:')
print(pd.DataFrame({'count': target_counts, 'pct': target_pct.round(1)}))

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['Completed (0)', 'Failed (1)'], target_counts.values,
       color=['#2ecc71', '#e74c3c'])
ax.set_title('Target Distribution — request_failed')
ax.set_ylabel('Count')
for i, v in enumerate(target_counts.values):
    ax.text(i, v + 100, f'{v:,}\n({target_pct.values[i]:.1f}%)', ha='center')
plt.tight_layout()
plt.savefig('reports/figures/target_distribution.png', dpi=150)
plt.show()
print('\n⚠️  Class imbalance: ~18% failed — keep this in mind for modelling (use stratified split, check precision/recall).')

## 5. Numerical Features — Summary Stats

In [ ]:
# SAFE features only (at_request)
num_features = ['distance_km', 'surge_multiplier', 'est_fare', 'eta_estimate_min']
trips[num_features].describe().round(3)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, num_features):
    trips[col].hist(bins=40, ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(col)
plt.suptitle('Numerical Features — Distributions', y=1.02)
plt.tight_layout()
plt.savefig('reports/figures/num_distributions.png', dpi=150)
plt.show()

## 6. Categorical Features — Value Counts

In [ ]:
cat_features = ['weather', 'payment_method', 'service_level']
for col in cat_features:
    print(f'\n{col}:')
    print(trips[col].value_counts(dropna=False).to_string())

## 7. Failure Rate by Category

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, cat_features):
    fail_rate = trips.groupby(col)['request_failed'].mean().sort_values(ascending=False)
    fail_rate.plot(kind='bar', ax=ax, color='salmon', edgecolor='white')
    ax.set_title(f'Failure Rate by {col}')
    ax.set_ylabel('Failure Rate')
    ax.set_ylim(0, 0.5)
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('reports/figures/failure_rate_by_category.png', dpi=150)
plt.show()

## 8. Temporal Patterns

In [ ]:
trips['hour']       = trips['request_ts'].dt.hour
trips['day_of_week']= trips['request_ts'].dt.dayofweek  # 0=Mon

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
trips.groupby('hour')['request_failed'].mean().plot(ax=axes[0], marker='o', color='coral')
axes[0].set_title('Failure Rate by Hour of Day')
axes[0].set_xlabel('Hour'); axes[0].set_ylabel('Failure Rate')

day_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
fail_by_day = trips.groupby('day_of_week')['request_failed'].mean()
fail_by_day.index = [day_names[i] for i in fail_by_day.index]
fail_by_day.plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Failure Rate by Day of Week')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('reports/figures/temporal_patterns.png', dpi=150)
plt.show()

# Clean up added cols
trips.drop(columns=['hour','day_of_week'], inplace=True)

## 9. Other Tables — Quick Look

In [ ]:
print('--- CUSTOMERS ---'); print(customers.head(3).to_string()); print(customers.dtypes)
print('\n--- DRIVERS ---'); print(drivers.head(3).to_string()); print(drivers.dtypes)
print('\nDrivers rating_avg missing:', drivers['rating_avg'].isnull().sum(), '(drivers with <min rated trips)')
print('\n--- ZONES ---'); print(zones.to_string())

## 10. Summary — Phase 0 Findings

In [ ]:
print('='*60)
print('PHASE 0 — DATASET SUMMARY')
print('='*60)
print(f'Train set (trips.csv):         {len(trips):,} rows')
print(f'Validation (recent_labelled):  {len(recent):,} rows')
print(f'Holdout (holdout_inputs.csv):  {len(holdout):,} rows')
print(f'Customers:  {len(customers):,} | Drivers: {len(drivers):,} | Zones: {len(zones):,}')
print()
print(f'Target: request_failed')
print(f'  0 (completed): {(trips.request_failed==0).sum():,} ({(trips.request_failed==0).mean()*100:.1f}%)')
print(f'  1 (failed):    {(trips.request_failed==1).sum():,} ({(trips.request_failed==1).mean()*100:.1f}%)')
print()
print('Safe features (at_request): 13 columns')
print('LEAKAGE columns (never use): actual_trip_time_min, cancel_reason, driver_arrival_min, support_ticket_flag')
print()
print('Missing values to handle in Phase 1:')
print('  - weather: 1,279 rows (4.5%)')
print('  - drivers.rating_avg: 264 drivers')
print()
print('✓ Phase 0 complete — ready for Phase 1 (Data Audit & Cleaning)')